In [0]:
-- cheapest payg vm per exact shape (vcpu + memory + cpu type + os + burstable or not) in paired regions
-- centralindia (pune) vs ap-south-1 (mumbai), eastus vs us-east-1. list prices only, cpu generation can differ
create or replace view cloud_pricing.gold.cross_cloud_compare as
with az as (
select p.region, case p.region when 'centralindia' then 'ap-south-1' when 'eastus' then 'us-east-1' end aws_region,
p.os, s.vcpus, s.memory_gb, s.cpu_arch, p.arm_sku_name, p.retail_price hourly,
-- B series is burstable (cpu credits), only compare with aws t family
p.arm_sku_name like 'Standard_B%' burstable
from cloud_pricing.silver.vm_prices p
join cloud_pricing.silver.azure_vm_specs s on s.region = p.region and s.sku_name = p.arm_sku_name
where p.observed_at = (select max(observed_at) from cloud_pricing.silver.vm_prices)
and p.price_type = 'Consumption' and p.capacity_type = 'regular' and p.is_primary_region
and p.retail_price > 0 and p.unit_of_measure = '1 Hour' and p.arm_sku_name not like '%Promo%'
),
aw as (
select region_code, os, vcpu, memory_gib, case when physical_processor like '%Graviton%' then 'arm64' else 'x64' end cpu_arch,
instance_type, effective_hourly hourly,
instance_type rlike '^t[0-9]' burstable
from cloud_pricing.silver.aws_ec2_prices
where price_model = 'on_demand' and effective_hourly > 0
and version = (select max(version) from cloud_pricing.silver.aws_ec2_prices)
),
az_best as (
select region, aws_region, os, vcpus, memory_gb, cpu_arch, burstable,
min_by(arm_sku_name, hourly) azure_cheapest, min(hourly) azure_hourly, count(*) azure_options
from az group by region, aws_region, os, vcpus, memory_gb, cpu_arch, burstable
),
aw_best as (
select region_code, os, vcpu, memory_gib, cpu_arch, burstable,
min_by(instance_type, hourly) aws_cheapest, min(hourly) aws_hourly, count(*) aws_options
from aw group by region_code, os, vcpu, memory_gib, cpu_arch, burstable
)
select a.region azure_region, a.aws_region, a.os, a.vcpus, a.memory_gb, a.cpu_arch, a.burstable,
a.azure_cheapest, a.azure_hourly, a.azure_options, w.aws_cheapest, w.aws_hourly, w.aws_options,
round((a.azure_hourly - w.aws_hourly) / w.aws_hourly * 100, 2) azure_vs_aws_pct
from az_best a
join aw_best w on w.region_code = a.aws_region and w.os = a.os and w.vcpu = a.vcpus
and w.memory_gib = a.memory_gb and w.cpu_arch = a.cpu_arch and w.burstable = a.burstable

In [0]:
select azure_region, os, vcpus, memory_gb, cpu_arch, burstable, azure_cheapest, azure_hourly, aws_cheapest, aws_hourly, azure_vs_aws_pct
from cloud_pricing.gold.cross_cloud_compare
where azure_region = 'centralindia' and os = 'linux' and vcpus in (2, 4, 8) and cpu_arch = 'x64'
order by vcpus, memory_gb, burstable